# ML Classification Stability — Residential ISP v2

Applies the **same saved XGBoost model** (trained on the anchor month) to four consecutive
snapshots and measures how binary predictions change over time.

Mimics `new_ml_tagging_final_access.ipynb` cells 35–36, 38, 45–50:

1. Load anchor model once (`./data/residential_isp_v2`)
2. Run inference on eyeball ASes for each month
3. Intersect ASes present in all months
4. Count raw vs hysteresis label flips
5. (Optional) Check Censys top-k feature changes for unstable ASes

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from as_tagging import ASTagging, OfflineSnapshotProvider, normalize_asn_input
from as_tagging.ml import MLTagger

# --- Config ---
DATA_PATH = "../as_feature_zenodo"
DATE_LIST = ["2026-01", "2026-02", "2026-03", "2026-04"]
ANCHOR_MONTH = "2026-01"  # month the saved model was trained on
MODEL_DIR = "./data/residential_isp_v2"
OUTPUT_DIR = "./data/stability"
THRESHOLD = 0.5
T_UP = 0.7       # hysteresis: rise to residential only above this
T_DOWN = 0.3     # hysteresis: fall to non-residential only below this


os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.isdir(MODEL_DIR):
    raise FileNotFoundError(
        f"Saved model not found at {MODEL_DIR}. "
        "Run the Residential ISP v2 cell in ml_tagging_example.ipynb first."
    )

provider = OfflineSnapshotProvider(DATA_PATH)
available = set(provider.list_snapshots())
missing = [d for d in DATE_LIST if d not in available]
if missing:
    raise FileNotFoundError(f"Snapshots not found in index.json: {missing}")
print(f"Months: {DATE_LIST}")
print(f"Anchor model: {MODEL_DIR} (trained on {ANCHOR_MONTH})")

## Step 1: Per-month inference (same model, different features)

In [ ]:
def eyeball_asns_for_month(date: str) -> list:
    tagger = ASTagging(snapshot_provider=provider, date=date, use_cache=True)
    return tagger.ListASNsWithoutTag("No Eyeball", treat_false_as_missing=True)


def predict_month(date: str, asns: list, threshold: float = THRESHOLD) -> pd.DataFrame:
    """Apply saved model to one month's snapshot; return prob/pred for requested ASNs."""
    tagger = ASTagging(snapshot_provider=provider, date=date, use_cache=True)

    manifest = snapshot_schema = None
    if hasattr(provider, "get_manifest"):
        try:
            manifest = provider.get_manifest(date)
        except Exception:
            pass
    if hasattr(provider, "get_schema"):
        try:
            snapshot_schema = provider.get_schema(date)
        except Exception:
            pass

    ml = MLTagger(
        snapshot_dict=tagger.atomic_tags,
        manifest=manifest,
        snapshot_schema=snapshot_schema,
        model_path=MODEL_DIR,
        verbose=False,
    )

    # Keep only ASNs present in this snapshot
    snapshot_keys = set(tagger.atomic_tags.keys())
    canon_to_key = {}
    for k in snapshot_keys:
        try:
            canon_to_key[normalize_asn_input(k)] = k
        except ValueError:
            continue

    present_asns = []
    for a in asns:
        canon = normalize_asn_input(a)
        if canon in canon_to_key:
            present_asns.append(canon_to_key[canon])

    probs = ml.predict(asns=present_asns)
    rows = []
    for asn, prob in probs.items():
        rows.append({
            "asn": int(normalize_asn_input(asn)),
            "month": date,
            "prob_xgb": float(prob),
            "pred_xgb": int(prob >= threshold),
            "thr": threshold,
        })
    return pd.DataFrame(rows)


# Eyeball sets per month
eyeball_by_month = {d: eyeball_asns_for_month(d) for d in DATE_LIST}
for d in DATE_LIST:
    print(f"{d}: {len(eyeball_by_month[d])} eyeball ASes")

# Intersect: present in eyeball scope in ALL months
eyeball_canon_sets = [
    {normalize_asn_input(a) for a in eyeball_by_month[d]} for d in DATE_LIST
]
common_eyeball = set.intersection(*eyeball_canon_sets)
print(f"\nEyeball ASes present in all {len(DATE_LIST)} months: {len(common_eyeball)}")

In [ ]:
frames = []
common_list = sorted(common_eyeball, key=lambda x: int(x))

for date in DATE_LIST:
    print(f"Inference on {date} ...", end=" ")
    df_month = predict_month(date, common_list, threshold=THRESHOLD)
    out_path = Path(OUTPUT_DIR) / f"probs_{date}.csv"
    df_month.to_csv(out_path, index=False)
    frames.append(df_month)
    n_pos = int((df_month["pred_xgb"] == 1).sum())
    print(f"{len(df_month)} ASes, {n_pos} predicted residential → {out_path}")

panel = pd.concat(frames, ignore_index=True)
print(f"\nPanel shape: {panel.shape}")

## Step 2: Stability analysis (raw vs hysteresis flips)

In [ ]:
def apply_hysteresis_for_as(probs, thrs, T_up=T_UP, T_down=T_DOWN):
    probs = np.asarray(probs, dtype=float)
    thrs = np.asarray(thrs, dtype=float)
    hyst = np.zeros_like(probs, dtype=int)
    hyst[0] = 1 if probs[0] >= thrs[0] else 0
    for i in range(1, len(probs)):
        prev, p = hyst[i - 1], probs[i]
        if prev == 1:
            hyst[i] = 1 if p >= T_down else 0
        else:
            hyst[i] = 0 if p <= T_up else 1
    return hyst


def count_flips(labels):
    labels = np.asarray(labels, dtype=int)
    if labels.size <= 1:
        return 0
    return int(np.sum(labels[1:] != labels[:-1]))


def collect_flips_for_as(asn, months, labels, label_type):
    labels = np.asarray(labels, dtype=int)
    flips = []
    for i in range(1, len(labels)):
        if labels[i] != labels[i - 1]:
            flips.append({
                "asn": asn,
                "label_type": label_type,
                "from_label": int(labels[i - 1]),
                "to_label": int(labels[i]),
                "month_from": months[i - 1],
                "month_to": months[i],
            })
    return flips


def analyze_stability(panel_df, date_list, T_up=T_UP, T_down=T_DOWN):
    date_list_sorted = sorted(date_list)
    order_map = {d: i for i, d in enumerate(date_list_sorted)}
    panel = panel_df.copy()
    panel["t_idx"] = panel["month"].map(order_map)
    panel = panel.sort_values(["asn", "t_idx"]).reset_index(drop=True)

    rows, hyst_labels_all, all_flips = [], [], []

    for asn, g in panel.groupby("asn", sort=False):
        g = g.sort_values("t_idx")
        months = g["month"].tolist()
        probs = g["prob_xgb"].to_numpy()
        thrs = g["thr"].to_numpy()
        raw = g["pred_xgb"].to_numpy()
        hyst = apply_hysteresis_for_as(probs, thrs, T_up=T_up, T_down=T_down)

        n_raw_flips = count_flips(raw)
        n_hyst_flips = count_flips(hyst)

        if n_raw_flips == 0:
            category = "stable_without_hysteresis"
        elif n_hyst_flips == 0:
            category = "stable_with_hysteresis"
        else:
            flips_cat = "1_flip" if n_hyst_flips == 1 else ("2_flips" if n_hyst_flips == 2 else f"{n_hyst_flips}_flips")
            category = f"unstable_after_hysteresis_{flips_cat}"

        rows.append({
            "asn": asn,
            "n_months": len(months),
            "n_raw_flips": n_raw_flips,
            "n_hyst_flips": n_hyst_flips,
            "category": category,
        })
        hyst_labels_all.append(pd.DataFrame({
            "asn": asn,
            "month": months,
            "t_idx": [order_map[m] for m in months],
            "raw_label": raw,
            "hyst_label": hyst,
            "prob_xgb": probs,
        }))
        all_flips.extend(collect_flips_for_as(asn, months, raw, "raw"))
        all_flips.extend(collect_flips_for_as(asn, months, hyst, "hysteresis"))

    summary_df = pd.DataFrame(rows).sort_values("asn").reset_index(drop=True)
    hyst_panel_df = pd.concat(hyst_labels_all, ignore_index=True)
    flips_df = pd.DataFrame(all_flips)

    total = len(summary_df)
    def pct(n): return 100.0 * n / max(1, total)

    n_stable_raw = (summary_df["category"] == "stable_without_hysteresis").sum()
    n_stable_hyst = (summary_df["category"] == "stable_with_hysteresis").sum()
    n_unstable = total - n_stable_raw - n_stable_hyst

    print(f"Total ASes (eyeball, all months): {total}")
    print(f"  Stable without hysteresis: {n_stable_raw} ({pct(n_stable_raw):.1f}%)")
    print(f"  Stable with hysteresis:    {n_stable_hyst} ({pct(n_stable_hyst):.1f}%)")
    print(f"  Unstable after hysteresis: {n_unstable} ({pct(n_unstable):.1f}%)")

    unstable = summary_df[summary_df["category"].str.startswith("unstable_after_hysteresis")]
    if not unstable.empty:
        print("\nUnstable breakdown:")
        print(unstable["category"].value_counts().to_string())

    return summary_df, hyst_panel_df, flips_df


summary_df, hyst_panel_df, flips_df = analyze_stability(panel, DATE_LIST)

summary_df.to_csv(Path(OUTPUT_DIR) / "stability_summary.csv", index=False)
hyst_panel_df.to_csv(Path(OUTPUT_DIR) / "monthly_labels.csv", index=False)
flips_df.to_csv(Path(OUTPUT_DIR) / "flip_events.csv", index=False)
print(f"\nSaved CSVs under {OUTPUT_DIR}/")

In [ ]:
# Category bar chart
cat_order = [
    "stable_without_hysteresis",
    "stable_with_hysteresis",
]
unstable_cats = sorted(
    [c for c in summary_df["category"].unique() if c.startswith("unstable")]
)
plot_cats = cat_order + unstable_cats
counts = summary_df["category"].value_counts().reindex(plot_cats).fillna(0)

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["#2ca02c", "#98df8a"] + ["#ff9896"] * len(unstable_cats)
counts.plot(kind="bar", ax=ax, color=colors[: len(plot_cats)])
ax.set_title(f"Classification stability ({DATE_LIST[0]} → {DATE_LIST[-1]}, n={len(summary_df)} eyeball ASes)")
ax.set_ylabel("Number of ASes")
ax.set_xlabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Step 3: Inspect individual AS trajectories

In [ ]:
hyst_flips_only = flips_df[flips_df["label_type"] == "hysteresis"]
ases_with_hyst_flips = hyst_flips_only["asn"].unique()
print(f"ASes with ≥1 hysteresis flip: {len(ases_with_hyst_flips)}")

# Example: first unstable AS
unstable = summary_df[summary_df["category"].str.startswith("unstable_after_hysteresis")]
if not unstable.empty:
    example_asn = int(unstable.iloc[0]["asn"])
    print(f"\nExample AS{example_asn} trajectory:")
    print(
        hyst_panel_df[hyst_panel_df["asn"] == example_asn]
        .sort_values("t_idx")[["month", "prob_xgb", "raw_label", "hyst_label"]]
        .to_string(index=False)
    )

## Step 4 (optional): Censys top-k changes for unstable ASes

For ASes that flip after hysteresis, check whether top Censys port/service/OS names changed
across months (feature drift vs threshold noise).

In [ ]:
TOP_KEYS = [
    "censys_port_1_name", "censys_port_2_name", "censys_port_3_name",
    "censys_service_1_name", "censys_service_2_name", "censys_service_3_name",
    "censys_os_1_name", "censys_os_2_name", "censys_os_3_name",
]


def load_snapshots(date_list):
    snaps = []
    for d in date_list:
        tagger = ASTagging(snapshot_provider=provider, date=d, use_cache=True)
        snaps.append(tagger.atomic_tags)
    return snaps


def summarize_censys_top_changes(snapshots, asn_list, snapshot_labels=None):
    if snapshot_labels is None:
        snapshot_labels = [f"t{i}" for i in range(len(snapshots))]
    rows = []
    for asn in asn_list:
        ports_all, srvs_all, oses_all = [], [], []
        seen = 0
        key_asn = str(asn)
        for snap in snapshots:
            key = key_asn if key_asn in snap else (int(asn) if int(asn) in snap else None)
            if key is None:
                continue
            seen += 1
            rec = snap[key]
            ports_all.append(tuple(rec.get(k) for k in TOP_KEYS[:3]))
            srvs_all.append(tuple(rec.get(k) for k in TOP_KEYS[3:6]))
            oses_all.append(tuple(rec.get(k) for k in TOP_KEYS[6:]))
        rows.append({
            "asn": asn,
            "n_snapshots_seen": seen,
            "changed_ports": len(set(ports_all)) > 1 if ports_all else False,
            "changed_services": len(set(srvs_all)) > 1 if srvs_all else False,
            "changed_os": len(set(oses_all)) > 1 if oses_all else False,
        })
    summary = pd.DataFrame(rows)
    summary["any_change"] = summary[["changed_ports", "changed_services", "changed_os"]].any(axis=1)
    return summary


if len(ases_with_hyst_flips) > 0:
    snapshots = load_snapshots(DATE_LIST)
    censys_summary = summarize_censys_top_changes(snapshots, ases_with_hyst_flips, DATE_LIST)
    n = len(censys_summary)
    n_any = int(censys_summary["any_change"].sum())
    print(f"Unstable ASes with any Censys top-k change: {n_any}/{n} ({100*n_any/max(1,n):.1f}%)")
    print(censys_summary.head(15).to_string(index=False))
else:
    print("No hysteresis flips — skipping Censys drift check.")